# Rushdey Model Validation Notebook

This notebook tests the graduation-project models outside the normal Flutter UI so we can answer a very specific question:

**Is the problem in the model itself, or in the mobile integration / camera / routing code?**

It validates:

- Wake word model: `wake_word_trial2.ptl`
- Currency models: YOLO detector `best.ptl` and classifier `currency_model.ptl`
- Face model: `face_model.tflite` when a TFLite runtime is available, plus an Android exact-model test path
- OCR: offline Arabic Tesseract data `ara.traineddata` when Python Tesseract is available, plus Android exact-model test path
- Vosk Arabic STT/intent model: when Python `vosk` is available

The notebook creates/uses sample data under `rushdey/static_test_images` and `rushdey/static_test_audio`.

## 0. Optional Dependency Install

Run this only if a section says a package is missing. The PyTorch currency and wake-word sections usually work without installing anything extra.

```bash
pip install pillow matplotlib librosa soundfile vosk pytesseract onnxruntime
```

For the TFLite face model on Python, one of these is needed:

```bash
pip install ai-edge-litert
# or, if available for your Python version:
pip install tflite-runtime
```

If Python TFLite/Tesseract is annoying on Windows, use the **Android exact-model test** section near the end. It uses the actual installed app engines on the phone.

In [ ]:
from pathlib import Path
import sys, os, json, math, subprocess, textwrap, shutil, wave

import numpy as np
from PIL import Image, ImageDraw, ImageFont
import matplotlib.pyplot as plt

try:
    from IPython.display import display
except Exception:
    display = print

# Notebook is saved in rushdey/. This finds repo root whether you run it from repo root or rushdey/.
HERE = Path.cwd().resolve()
if (HERE / 'assets' / 'models').exists() and HERE.name.lower() == 'rushdey':
    APP_ROOT = HERE
    REPO_ROOT = HERE.parent
elif (HERE / 'rushdey' / 'assets' / 'models').exists():
    REPO_ROOT = HERE
    APP_ROOT = HERE / 'rushdey'
else:
    raise RuntimeError('Run this notebook from the repo root or from the rushdey folder.')

MODELS_DIR = APP_ROOT / 'assets' / 'models'
TESSDATA_DIR = APP_ROOT / 'assets' / 'tessdata'
VOSK_DIR = APP_ROOT / 'android' / 'app' / 'src' / 'main' / 'assets' / 'vosk-model-ar'
STATIC_IMAGES = APP_ROOT / 'static_test_images'
STATIC_AUDIO = APP_ROOT / 'static_test_audio'

paths = {
    'repo_root': REPO_ROOT,
    'app_root': APP_ROOT,
    'models_dir': MODELS_DIR,
    'static_images': STATIC_IMAGES,
    'static_audio': STATIC_AUDIO,
}
for k, v in paths.items():
    print(f'{k}: {v}')

required_files = [
    MODELS_DIR / 'wake_word_trial2.ptl',
    MODELS_DIR / 'best.ptl',
    MODELS_DIR / 'currency_model.ptl',
    MODELS_DIR / 'face_model.tflite',
    TESSDATA_DIR / 'ara.traineddata',
]
for p in required_files:
    print(('OK   ' if p.exists() else 'MISS '), p)

In [ ]:
def dependency_status():
    names = ['torch', 'cv2', 'librosa', 'soundfile', 'vosk', 'pytesseract', 'tensorflow', 'tflite_runtime', 'ai_edge_litert']
    status = {}
    for name in names:
        try:
            mod = __import__(name)
            status[name] = f'OK {getattr(mod, "__version__", "")}'.strip()
        except Exception as e:
            status[name] = f'MISSING/ERROR: {type(e).__name__}: {str(e)[:100]}'
    return status

status = dependency_status()
for k, v in status.items():
    print(f'{k:15s} {v}')

## 1. Build / Download Sample Data

These are not training samples. They are small sanity-check samples:

- OCR: generated Arabic text image and blank image
- Currency: real Egyptian banknote images plus a blank negative
- Face: public portraits for positive/negative recognition tests
- Audio: generated silence/noise negatives, plus folders where you can put real wake-word or Vosk `.wav` samples

In [ ]:
import urllib.request

STATIC_IMAGES.mkdir(parents=True, exist_ok=True)
(STATIC_IMAGES / 'ocr').mkdir(parents=True, exist_ok=True)
(STATIC_IMAGES / 'currency').mkdir(parents=True, exist_ok=True)
(STATIC_IMAGES / 'face' / 'enroll' / 'obama').mkdir(parents=True, exist_ok=True)
(STATIC_IMAGES / 'face' / 'query' / 'obama').mkdir(parents=True, exist_ok=True)
(STATIC_IMAGES / 'face' / 'query' / 'not_obama').mkdir(parents=True, exist_ok=True)
(STATIC_AUDIO / 'wake_positive').mkdir(parents=True, exist_ok=True)
(STATIC_AUDIO / 'wake_negative').mkdir(parents=True, exist_ok=True)
(STATIC_AUDIO / 'vosk').mkdir(parents=True, exist_ok=True)

def make_arabic_image(path, lines=None, blank=False):
    path = Path(path)
    if path.exists():
        return
    img = Image.new('RGB', (1200, 800), 'white')
    if not blank:
        draw = ImageDraw.Draw(img)
        font_candidates = [
            Path('C:/Windows/Fonts/tahoma.ttf'),
            Path('C:/Windows/Fonts/arial.ttf'),
            Path('C:/Windows/Fonts/arabtype.ttf'),
        ]
        font_path = next((p for p in font_candidates if p.exists()), None)
        font = ImageFont.truetype(str(font_path), 72) if font_path else ImageFont.load_default()
        y = 140
        for line in (lines or ['????? ????', '???? ??? ????', '?????? 100 ????']):
            # Keep it simple: PIL on this machine may not have full Arabic shaping, but Tesseract still gets a usable sample.
            try:
                draw.text((80, y), line, fill='black', font=font, direction='rtl')
            except Exception:
                draw.text((80, y), line, fill='black', font=font)
            y += 110
    img.save(path)

make_arabic_image(STATIC_IMAGES / 'ocr' / 'arabic_text.png')
make_arabic_image(STATIC_IMAGES / 'ocr' / 'blank.png', blank=True)
# Also use blank image as a currency negative.
if not (STATIC_IMAGES / 'currency' / 'no_currency_blank.png').exists():
    shutil.copy2(STATIC_IMAGES / 'ocr' / 'blank.png', STATIC_IMAGES / 'currency' / 'no_currency_blank.png')

samples = {
    STATIC_IMAGES / 'currency' / 'egp_100_front.jpg': 'https://commons.wikimedia.org/wiki/Special:FilePath/EGP%20100%20Pounds%202015%20%28Front%29.jpg',
    STATIC_IMAGES / 'currency' / 'egp_100_obverse_2014.jpg': 'https://commons.wikimedia.org/wiki/Special:FilePath/100%20EGP%20obverse%202014-1-26.jpg',
    STATIC_IMAGES / 'currency' / 'egp_200_front.jpg': 'https://commons.wikimedia.org/wiki/Special:FilePath/EGP%20200%20Pounds%20Apr%202007%20%28Front%29.jpg',
    STATIC_IMAGES / 'currency' / 'egp_50_obverse_2011.jpg': 'https://commons.wikimedia.org/wiki/Special:FilePath/50%20EGP%20obverse%202011-5-4.jpg',
    STATIC_IMAGES / 'face' / 'enroll' / 'obama' / 'obama_enroll.jpg': 'https://commons.wikimedia.org/wiki/Special:FilePath/Official%20portrait%20of%20Barack%20Obama.jpg',
    STATIC_IMAGES / 'face' / 'query' / 'obama' / 'obama_query.jpg': 'https://commons.wikimedia.org/wiki/Special:FilePath/President%20Barack%20Obama%20%281%29.jpg',
    STATIC_IMAGES / 'face' / 'query' / 'not_obama' / 'biden_query.jpg': 'https://commons.wikimedia.org/wiki/Special:FilePath/Joe%20Biden%20presidential%20portrait.jpg',
}

for out_path, url in samples.items():
    if out_path.exists() and out_path.stat().st_size > 10_000:
        print('exists', out_path.name)
        continue
    print('downloading', out_path.name)
    req = urllib.request.Request(url, headers={'User-Agent': 'RushdeyModelValidation/1.0'})
    with urllib.request.urlopen(req, timeout=60) as response:
        data = response.read()
    if len(data) < 10_000:
        raise RuntimeError(f'Download too small for {out_path}: {len(data)} bytes')
    out_path.write_bytes(data)

# Generate simple negative audio clips for wake-word sanity checks.
import soundfile as sf
sr = 16000
if not (STATIC_AUDIO / 'wake_negative' / 'silence.wav').exists():
    sf.write(STATIC_AUDIO / 'wake_negative' / 'silence.wav', np.zeros(sr, dtype=np.float32), sr)
if not (STATIC_AUDIO / 'wake_negative' / 'noise.wav').exists():
    rng = np.random.default_rng(42)
    sf.write(STATIC_AUDIO / 'wake_negative' / 'noise.wav', (0.015 * rng.standard_normal(sr)).astype(np.float32), sr)

print('Sample image count:', len(list(STATIC_IMAGES.rglob('*.*'))))
print('Sample audio count:', len(list(STATIC_AUDIO.rglob('*.wav'))))

In [ ]:
def show_images(paths, cols=3, figsize=(12, 7)):
    paths = [Path(p) for p in paths]
    rows = math.ceil(len(paths) / cols)
    plt.figure(figsize=figsize)
    for i, p in enumerate(paths, 1):
        plt.subplot(rows, cols, i)
        img = Image.open(p).convert('RGB')
        plt.imshow(img)
        plt.title(p.name, fontsize=9)
        plt.axis('off')
    plt.tight_layout()

show_images(list((STATIC_IMAGES / 'currency').glob('*.*'))[:6], cols=3)

## 2. Currency Model Validation

This tests both currency models:

- `best.ptl`: YOLO-style detector, traced at **320x320**
- `currency_model.ptl`: 9-class classifier, good fallback for clean/full banknote images

Expected behavior on the included samples:

- `egp_50...` -> 50
- `egp_100...` -> 100
- `egp_200...` -> 200
- `no_currency_blank.png` -> no currency

In [ ]:
import torch
import torch.nn.functional as F

classifier_labels = ['1_EGP','5_EGP','10_EGP','10_EGP_NEW','20_EGP','20_EGP_NEW','50_EGP','100_EGP','200_EGP']
detector_labels = ['1_EGP','5_EGP','10_EGP','20_EGP','50_EGP','100_EGP','200_EGP']
values = {
    '1_EGP': 1, '5_EGP': 5, '10_EGP': 10, '10_EGP_NEW': 10,
    '20_EGP': 20, '20_EGP_NEW': 20, '50_EGP': 50, '100_EGP': 100, '200_EGP': 200,
}
expected_currency = {
    'egp_50_obverse_2011.jpg': 50,
    'egp_100_front.jpg': 100,
    'egp_100_obverse_2014.jpg': 100,
    'egp_200_front.jpg': 200,
    'no_currency_blank.png': 0,
}

def pil_to_classifier_tensor(path, size=224):
    img = Image.open(path).convert('RGB').resize((size, size))
    arr = np.asarray(img).astype(np.float32) / 255.0
    arr = np.transpose(arr, (2, 0, 1))
    mean = np.array([0.485, 0.456, 0.406], dtype=np.float32).reshape(3, 1, 1)
    std = np.array([0.229, 0.224, 0.225], dtype=np.float32).reshape(3, 1, 1)
    arr = (arr - mean) / std
    return torch.from_numpy(arr).unsqueeze(0)

def pil_to_detector_tensor(path, size=320):
    img = Image.open(path).convert('RGB').resize((size, size))
    arr = np.asarray(img).astype(np.float32) / 255.0
    arr = np.transpose(arr, (2, 0, 1))
    return torch.from_numpy(arr).unsqueeze(0)

def run_currency_classifier(path):
    model = torch.jit.load(str(MODELS_DIR / 'currency_model.ptl'), map_location='cpu').eval()
    with torch.no_grad():
        logits = model(pil_to_classifier_tensor(path))
        probs = F.softmax(logits, dim=1)[0]
    idx = int(torch.argmax(probs).item())
    label = classifier_labels[idx]
    conf = float(probs[idx].item())
    value = values[label] if conf >= 0.70 else 0
    return {'label': label, 'confidence': conf, 'value': value}

def run_currency_detector(path, threshold=0.35):
    model = torch.jit.load(str(MODELS_DIR / 'best.ptl'), map_location='cpu').eval()
    with torch.no_grad():
        y = model(pil_to_detector_tensor(path, 320))
    data = y[0].numpy() if isinstance(y, torch.Tensor) else y[0][0].numpy()
    # Shape from model: [1, 11, 2100] => [11, 2100]
    if data.ndim == 3:
        data = data[0]
    num_anchors = data.shape[1]
    detections = []
    for i in range(num_anchors):
        scores = data[4:4+len(detector_labels), i]
        best_idx = int(np.argmax(scores))
        best_score = float(scores[best_idx])
        if best_score >= threshold:
            label = detector_labels[best_idx]
            detections.append((label, best_score, values[label]))
    if not detections:
        return {'detections': [], 'total': 0}
    # For this sanity test, top detection is enough; Android app applies NMS.
    detections = sorted(detections, key=lambda x: x[1], reverse=True)[:5]
    return {'detections': detections, 'total': sum(d[2] for d in detections[:1])}

rows = []
for p in sorted((STATIC_IMAGES / 'currency').glob('*.*')):
    if p.suffix.lower() not in ['.jpg', '.jpeg', '.png']:
        continue
    clf = run_currency_classifier(p)
    det = run_currency_detector(p)
    detector_value = det['total']
    final_value = detector_value if detector_value > 0 else clf['value']
    expected = expected_currency.get(p.name, None)
    rows.append({
        'file': p.name,
        'expected': expected,
        'detector_total': detector_value,
        'classifier': f"{clf['label']} {clf['confidence']:.3f}",
        'final_value': final_value,
        'pass': (expected is None or expected == final_value),
    })

for r in rows:
    print(r)
print('\nCurrency pass:', all(r['pass'] for r in rows))

## 3. Wake Word Model Validation

This validates that the wake-word TorchScript model loads and produces sane probabilities.

Important: the notebook can only do a **positive** wake-word test if you provide real recordings of ??????. Put `.wav` files here:

`rushdey/static_test_audio/wake_positive/`

The notebook already creates negative samples under:

`rushdey/static_test_audio/wake_negative/`

In [ ]:
import librosa

WAKE_MODEL = MODELS_DIR / 'wake_word_trial2.ptl'

sample_rate = 16000
wake_cfg = {
    'sr': 16000,
    'duration': 1.0,
    'n_fft': 1024,
    'hop_length': 512,
    'n_mels': 64,
    'fmin': 20,
    'fmax': 8000,
    'frames': 32,
}

def load_audio_1s(path):
    y, sr0 = librosa.load(path, sr=wake_cfg['sr'], mono=True)
    n = int(wake_cfg['sr'] * wake_cfg['duration'])
    if len(y) < n:
        y = np.pad(y, (0, n - len(y)))
    else:
        y = y[:n]
    return y.astype(np.float32)

def audio_to_mel_tensor(y):
    mel = librosa.feature.melspectrogram(
        y=y,
        sr=wake_cfg['sr'],
        n_fft=wake_cfg['n_fft'],
        hop_length=wake_cfg['hop_length'],
        n_mels=wake_cfg['n_mels'],
        fmin=wake_cfg['fmin'],
        fmax=wake_cfg['fmax'],
        power=2.0,
        center=True,
    )
    db = librosa.power_to_db(mel, ref=np.max, top_db=80.0)
    db_min, db_max = float(db.min()), float(db.max())
    mel_norm = (db - db_min) / max(db_max - db_min, 1e-6)
    # force [64, 32]
    if mel_norm.shape[1] < wake_cfg['frames']:
        mel_norm = np.pad(mel_norm, ((0,0), (0, wake_cfg['frames'] - mel_norm.shape[1])))
    else:
        mel_norm = mel_norm[:, :wake_cfg['frames']]
    return torch.from_numpy(mel_norm.astype(np.float32)).unsqueeze(0).unsqueeze(0)

def predict_wake(path):
    model = torch.jit.load(str(WAKE_MODEL), map_location='cpu').eval()
    y = load_audio_1s(path)
    rms = float(np.sqrt(np.mean(y * y)))
    x = audio_to_mel_tensor(y)
    with torch.no_grad():
        logits = model(x)
        probs = F.softmax(logits, dim=1)[0].cpu().numpy()
    wake_prob = float(probs[1])
    # Mirrors the mobile guard: do not trigger on silence/background even if raw model probability is high.
    app_detect = wake_prob >= 0.50 and rms >= 0.025
    return {'negative_prob': float(probs[0]), 'wake_prob': wake_prob, 'rms': rms, 'app_detect': app_detect}

wake_rows = []
for folder, expected in [('wake_negative', 'negative'), ('wake_positive', 'wake')]:
    for p in sorted((STATIC_AUDIO / folder).glob('*.wav')):
        pred = predict_wake(p)
        label = 'wake' if pred['app_detect'] else 'negative'
        wake_rows.append({'file': str(p.relative_to(STATIC_AUDIO)), 'expected': expected, 'pred': label, **pred})

if not wake_rows:
    print('No wake audio files found.')
else:
    for r in wake_rows:
        print(r)

if not list((STATIC_AUDIO / 'wake_positive').glob('*.wav')):
    print('\nNOTE: Add real recordings of ???? to static_test_audio/wake_positive/ for a true positive test.')

## 4. Face Model Validation

The mobile app uses `face_model.tflite` plus ML Kit face detection/cropping. Python cannot use ML Kit directly.

This section tries a Python TFLite runtime if available. If no TFLite runtime is installed, use the Android exact-model test section below; that path uses the actual Android app code and ML Kit detector.

In [ ]:
def get_tflite_interpreter():
    # Try ai-edge-litert first.
    try:
        from ai_edge_litert.interpreter import Interpreter
        return Interpreter
    except Exception:
        pass
    try:
        from tflite_runtime.interpreter import Interpreter
        return Interpreter
    except Exception:
        pass
    try:
        import tensorflow as tf
        return tf.lite.Interpreter
    except Exception as e:
        print('No usable TFLite runtime:', repr(e))
        return None

Interpreter = get_tflite_interpreter()
if Interpreter is None:
    print('SKIP face Python test. Install ai-edge-litert or use Android exact-model test below.')
else:
    import cv2
    face_cascade = cv2.CascadeClassifier(str(Path(cv2.data.haarcascades) / 'haarcascade_frontalface_default.xml'))
    interpreter = Interpreter(model_path=str(MODELS_DIR / 'face_model.tflite'))
    interpreter.allocate_tensors()
    input_info = interpreter.get_input_details()[0]
    output_info = interpreter.get_output_details()[0]
    print('TFLite input:', input_info['shape'], input_info['dtype'])
    print('TFLite output:', output_info['shape'], output_info['dtype'])

    def crop_largest_face(path):
        img = cv2.imread(str(path))
        if img is None:
            raise RuntimeError(f'Could not read {path}')
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        faces = face_cascade.detectMultiScale(gray, scaleFactor=1.1, minNeighbors=5)
        if len(faces) == 0:
            raise RuntimeError(f'No face detected: {path.name}')
        x, y, w, h = max(faces, key=lambda b: b[2]*b[3])
        pad = int(0.2 * max(w, h))
        x1, y1 = max(0, x-pad), max(0, y-pad)
        x2, y2 = min(img.shape[1], x+w+pad), min(img.shape[0], y+h+pad)
        crop = img[y1:y2, x1:x2]
        crop = cv2.resize(crop, (112, 112))
        crop = cv2.cvtColor(crop, cv2.COLOR_BGR2RGB).astype(np.float32)
        crop = crop / 127.5 - 1.0
        return crop

    def face_embedding(path):
        crop = crop_largest_face(path)
        x = np.expand_dims(crop, axis=0).astype(input_info['dtype'])
        interpreter.set_tensor(input_info['index'], x)
        interpreter.invoke()
        emb = interpreter.get_tensor(output_info['index'])[0].astype(np.float32)
        emb = emb / max(np.linalg.norm(emb), 1e-6)
        return emb

    enroll = STATIC_IMAGES / 'face' / 'enroll' / 'obama' / 'obama_enroll.jpg'
    q_pos = STATIC_IMAGES / 'face' / 'query' / 'obama' / 'obama_query.jpg'
    q_neg = STATIC_IMAGES / 'face' / 'query' / 'not_obama' / 'biden_query.jpg'
    e = face_embedding(enroll)
    for p in [q_pos, q_neg]:
        sim = float(np.dot(e, face_embedding(p)))
        print({'query': p.name, 'similarity': sim, 'recognized_at_0.985': sim >= 0.985})

## 5. OCR Validation

Python OCR needs `pytesseract` plus the Tesseract executable installed on Windows. If unavailable, use the Android exact-model test below; it uses the app's bundled `ara.traineddata` through Tesseract4Android.

In [ ]:
try:
    import pytesseract
    ocr_available = True
except Exception as e:
    print('SKIP Python OCR: pytesseract unavailable:', repr(e))
    ocr_available = False

if ocr_available:
    for p in sorted((STATIC_IMAGES / 'ocr').glob('*.*')):
        text = pytesseract.image_to_string(Image.open(p), lang='ara', config='--psm 6')
        print({'file': p.name, 'text': text.strip().replace('\n', ' | ')})
else:
    print('Use Android exact-model test section for OCR.')

## 6. Vosk STT / Intent Validation

Python Vosk is optional. If installed, this section loads the bundled Arabic Vosk model and runs any `.wav` files you put here:

`rushdey/static_test_audio/vosk/`

The intent mapping mirrors the app: face, OCR, currency, unknown.

In [ ]:
def detect_intent_ar(text):
    t = (text or '').strip().lower()
    face_words = ['???', '??? ??', '??? ??', '??? ?????', '?? ?????', '?? ???']
    ocr_words = ['?????', '????', '??????', '????', '???????']
    currency_words = ['???', '??', '??? ???', '??? ??', '???', '?? ???']
    if any(w in t for w in face_words):
        return 'face_who_is_in_front'
    if any(w in t for w in ocr_words):
        return 'ocr_read_text'
    if any(w in t for w in currency_words):
        return 'currency_count'
    return 'unknown'

try:
    from vosk import Model, KaldiRecognizer
    vosk_available = True
except Exception as e:
    print('SKIP Vosk Python test:', repr(e))
    vosk_available = False

if vosk_available:
    if not VOSK_DIR.exists():
        print('Vosk model folder missing:', VOSK_DIR)
    else:
        model = Model(str(VOSK_DIR))
        wavs = sorted((STATIC_AUDIO / 'vosk').glob('*.wav'))
        if not wavs:
            print('No Vosk wav samples. Put Arabic command .wav files in:', STATIC_AUDIO / 'vosk')
        for wav_path in wavs:
            wf = wave.open(str(wav_path), 'rb')
            rec = KaldiRecognizer(model, wf.getframerate())
            while True:
                data = wf.readframes(4000)
                if not data:
                    break
                rec.AcceptWaveform(data)
            result = json.loads(rec.FinalResult())
            text = result.get('text', '')
            print({'file': wav_path.name, 'text': text, 'intent': detect_intent_ar(text)})

## 7. Android Exact-Model Test (Recommended For Final App Evidence)

This uses the debug receiver inside the Android app to run the actual Android engines against the same static images:

- Android Tesseract OCR
- Android PyTorch currency detector/classifier
- Android TFLite face embedding + ML Kit detector

Requirements:

- Phone connected with USB debugging
- Debug APK installed
- `adb` available through Android SDK

This is the closest test to the final mobile app without using the camera.

In [ ]:
def find_adb():
    candidates = []
    for env in ['ANDROID_HOME', 'ANDROID_SDK_ROOT']:
        if os.environ.get(env):
            candidates.append(Path(os.environ[env]) / 'platform-tools' / 'adb.exe')
            candidates.append(Path(os.environ[env]) / 'platform-tools' / 'adb')
    candidates += [
        Path.home() / 'AppData' / 'Local' / 'Android' / 'Sdk' / 'platform-tools' / 'adb.exe',
        Path('/usr/bin/adb'),
    ]
    for p in candidates:
        if p.exists():
            return str(p)
    return shutil.which('adb')

adb = find_adb()
print('adb:', adb)

if not adb:
    print('ADB not found. Skip Android exact test.')
else:
    # Copy static images into app internal storage through run-as.
    subprocess.run([adb, 'devices'], check=False)
    subprocess.run([adb, 'shell', 'rm', '-rf', '/data/local/tmp/rushdey_static_test_images'], check=False)
    subprocess.run([adb, 'push', str(STATIC_IMAGES), '/data/local/tmp/rushdey_static_test_images'], check=True)
    subprocess.run([adb, 'shell', 'run-as', 'com.example.rushdey', 'rm', '-rf', 'files/test_images'], check=False)
    subprocess.run([adb, 'shell', 'run-as', 'com.example.rushdey', 'cp', '-R', '/data/local/tmp/rushdey_static_test_images', 'files/test_images'], check=True)
    subprocess.run([adb, 'logcat', '-c'], check=False)
    subprocess.run([
        adb, 'shell', 'am', 'broadcast',
        '-n', 'com.example.rushdey/.StaticImageTestReceiver',
        '-a', 'com.example.rushdey.RUN_IMAGE_TESTS'
    ], check=True)
    import time
    time.sleep(8)
    logs = subprocess.check_output([
        adb, 'logcat', '-d', '-s',
        'RushdeyStaticTest:I', 'OCREngine:D', 'CurrencyEngine:I', 'FaceEngine:I', 'FaceEngine:W', 'AndroidRuntime:E', '*:S'
    ], text=True, errors='replace')
    lines = [line for line in logs.splitlines() if 'RUSHDEY_STATIC_TEST' in line]
    print('\n'.join(lines))

## 8. Summary Checklist

Use this checklist after running all relevant cells:

- Currency classifier detects 50/100/200 and rejects blank image.
- Currency detector runs at 320x320 without anchor mismatch errors.
- Face positive pair similarity is above the threshold; negative pair is below threshold.
- OCR blank image says no clear text instead of random Arabic/digits.
- Wake word negatives are below threshold; real ?????? samples are above threshold.
- Vosk command samples map to the expected intent.

If a Python section is skipped because of dependencies, run the Android exact-model test. That section validates the same engines used by the phone app.